# MeterMind — Model 3: LLM Prompting (Claude)

This notebook implements the **large pretrained LLM baseline** for MeterMind.

Rather than training anything, we simply prompt Claude to rewrite expanded archaic prose into iambic pentameter — zero-shot, using only its pretrained knowledge of poetry and meter.

### Why this is interesting
Claude has been trained on vast amounts of text including poetry, so it has implicit knowledge of iambic meter, Shakespearean register, and stress patterns — knowledge that neither the DP reorderer nor the GRU has access to. The question is whether that broad pretrained knowledge outperforms a task-specific trained model (GRU) or a rule-based symbolic system (DP).

### Pipeline
```
test pairs → Claude API (prompted) → evaluate (MA, SP, G) → compare with Models 1 & 2
```

## 0. Install dependencies

In [ ]:
%pip install anthropic pronouncing sentence-transformers --quiet

## 1. Imports & configuration

In [ ]:
import re, io, os, json, time, math
import pandas as pd
import matplotlib.pyplot as plt
import torch
import pronouncing
import anthropic
from google.colab import files
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer, util
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

# ── API ───────────────────────────────────────────────────────────────────────
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY", "YOUR_KEY_HERE"))

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL      = "claude-haiku-4-5-20251001"   # cheap & fast; swap for claude-sonnet-4-6 for higher quality
MAX_TOKENS = 100                            # output is a single short line
RETRY_LIMIT   = 3
SLEEP_BETWEEN = 0.3

print('Config ready.')

## 2. Load test set

Upload `training_pairs.csv` — we'll take the same 10% test split as Model 2 for a fair comparison.

**Important:** use the same random seed so the test split is identical across all three models.

In [ ]:
import random
SEED = 42
random.seed(SEED)

uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.StringIO(uploaded[filename].decode('utf-8')))
pairs = list(zip(df['input'].tolist(), df['target'].tolist()))
random.shuffle(pairs)

# Replicate exact same split as Model 2
unique_targets = list({t for _, t in pairs})
random.shuffle(unique_targets)
n = len(unique_targets)
test_targets = set(unique_targets[int(n * 0.9):])
test_pairs   = [(s, t) for s, t in pairs if t in test_targets]

print(f'Test pairs: {len(test_pairs):,}')
print(f'\nExample:')
print(f'  INPUT  : {test_pairs[0][0]}')
print(f'  TARGET : {test_pairs[0][1]}')

## 3. Prompt design

The prompt is the core of this model. It needs to:
1. Tell Claude exactly what iambic pentameter is
2. Constrain it to only use words from the input
3. Preserve archaic register
4. Return only the rewritten line — no explanation

We use a **zero-shot** prompt — no examples provided. This tests Claude's pretrained knowledge directly.

In [ ]:
SYSTEM_PROMPT = """\
You are an expert in Shakespearean poetry and iambic pentameter.

Iambic pentameter is a line of 10 syllables with alternating unstressed and stressed beats:
  da-DUM da-DUM da-DUM da-DUM da-DUM

Your task: rewrite the given archaic prose line into iambic pentameter.

STRICT RULES:
1. Use ONLY words that appear in the input line — do not add new words.
2. You may reorder words freely.
3. Preserve the archaic register (thee, thou, thy, doth, hath, etc.).
4. Aim for exactly 10 syllables matching the da-DUM pattern.
5. Return ONLY the rewritten line — no explanation, no punctuation changes, no quotation marks.
"""

USER_TEMPLATE = "Rewrite this into iambic pentameter:\n{line}"


def prompt_claude(line):
    """Send a single line to Claude and return the rewritten version."""
    for attempt in range(RETRY_LIMIT):
        try:
            response = client.messages.create(
                model=MODEL,
                max_tokens=MAX_TOKENS,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": USER_TEMPLATE.format(line=line)}]
            )
            return response.content[0].text.strip()
        except Exception as e:
            print(f'  Attempt {attempt+1} failed: {e}')
            time.sleep(2 ** attempt)
    return None


# Smoke test
test_out = prompt_claude(test_pairs[0][0])
print(f'INPUT  : {test_pairs[0][0]}')
print(f'TARGET : {test_pairs[0][1]}')
print(f'OUTPUT : {test_out}')

## 4. Evaluation metrics

Same three metrics as Models 1 and 2.

In [ ]:
IAMBIC_TEMPLATE = [0, 1] * 10

def tokenize(text):
    return re.sub(r"[^a-zA-Z\s]", "", text.lower()).split()

def get_stress(word):
    phones = pronouncing.phones_for_word(word)
    if not phones:
        for suffix in ['est', 'eth']:
            if word.endswith(suffix) and len(word) > len(suffix) + 1:
                root_phones = pronouncing.phones_for_word(word[:-len(suffix)])
                if root_phones:
                    stress = [1 if s == '2' else int(s)
                              for s in pronouncing.stresses(root_phones[0]) if s in '012']
                    return stress + [0]
        return [0]
    stress = [1 if s == '2' else int(s)
              for s in pronouncing.stresses(phones[0]) if s in '012']
    if len(stress) == 1 and stress[0] == 1:
        return [0, 1]
    return stress

def n_syllables(word):
    phones = pronouncing.phones_for_word(word)
    if not phones:
        return 1
    return len([p for p in phones[0].split() if p[-1].isdigit()])

def is_flexible(word):
    return get_stress(word) == [0, 1]

def metrical_accuracy(words, template=IAMBIC_TEMPLATE):
    correct, syl_pos = 0, 0
    for word in words:
        if syl_pos >= len(template): break
        if is_flexible(word):
            correct += 1; syl_pos += 1
        else:
            for s in get_stress(word):
                if syl_pos >= len(template): break
                if s == template[syl_pos]: correct += 1
                syl_pos += 1
    return correct / len(template)

# Semantic Preservation
print('Loading SentenceTransformer...')
sp_model = SentenceTransformer('all-MiniLM-L6-v2')

def semantic_preservation(original, output):
    emb = sp_model.encode([original, output], convert_to_tensor=True)
    return float(util.cos_sim(emb[0], emb[1]))

# Grammaticality
print('Loading GPT-2...')
gpt2_tok   = GPT2TokenizerFast.from_pretrained('gpt2')
gpt2_model = GPT2LMHeadModel.from_pretrained('gpt2')
gpt2_model.eval()

def grammaticality(text):
    inputs = gpt2_tok(text, return_tensors='pt')
    with torch.no_grad():
        loss = gpt2_model(**inputs, labels=inputs['input_ids']).loss
    return 1 / (1 + math.log(math.exp(loss.item())))

print('All metrics ready.')

## 5. Run on test set

In [ ]:
results  = []
failures = []

for src, tgt in tqdm(test_pairs, desc='Prompting Claude'):
    output = prompt_claude(src)

    if output is None:
        failures.append(src)
        continue

    words = tokenize(output)
    ma = metrical_accuracy(words)
    sp = semantic_preservation(src, output)
    g  = grammaticality(output)

    results.append({
        'input':  src,
        'target': tgt,
        'output': output,
        'MA':     round(ma, 4),
        'SP':     round(sp, 4),
        'G':      round(g,  4),
    })

    time.sleep(SLEEP_BETWEEN)

results_df = pd.DataFrame(results)
print(f'Done. {len(results_df):,} results, {len(failures)} failures.')

## 6. Results

In [ ]:
print('=== Claude (LLM) — Test Set Results ===')
print(f"Metrical Accuracy (MA)    : {results_df['MA'].mean():.4f} ± {results_df['MA'].std():.4f}")
print(f"Semantic Preservation (SP): {results_df['SP'].mean():.4f} ± {results_df['SP'].std():.4f}")
print(f"Grammaticality (G)        : {results_df['G'].mean():.4f} ± {results_df['G'].std():.4f}")

print('\n=== Sample outputs ===')
for _, row in results_df.head(5).iterrows():
    print(f"INPUT  : {row['input']}")
    print(f"TARGET : {row['target']}")
    print(f"OUTPUT : {row['output']}")
    print(f"MA={row['MA']:.3f}  SP={row['SP']:.3f}  G={row['G']:.3f}")
    print()

# Distribution plots
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Claude (LLM) — Metric Distributions', fontsize=14)
for ax, metric, colour in zip(axes, ['MA', 'SP', 'G'], ['steelblue', 'seagreen', 'tomato']):
    ax.hist(results_df[metric], bins=20, color=colour, edgecolor='white', alpha=0.85)
    ax.axvline(results_df[metric].mean(), color='black', linestyle='--', linewidth=1.5,
               label=f"mean={results_df[metric].mean():.3f}")
    ax.set_title(metric)
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')
    ax.legend()
plt.tight_layout()
plt.savefig('llm_metrics.png', dpi=150)
plt.show()

## 7. Save results

In [ ]:
results_df.to_csv('llm_results.csv', index=False)
print('Saved llm_results.csv')

summary = {
    'model': 'Claude (LLM)',
    'n':     len(results_df),
    'MA':    {'mean': round(float(results_df['MA'].mean()), 4), 'std': round(float(results_df['MA'].std()), 4)},
    'SP':    {'mean': round(float(results_df['SP'].mean()), 4), 'std': round(float(results_df['SP'].std()), 4)},
    'G':     {'mean': round(float(results_df['G'].mean()),  4), 'std': round(float(results_df['G'].std()),  4)},
}
with open('llm_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved llm_summary.json')

files.download('llm_results.csv')
files.download('llm_summary.json')
files.download('llm_metrics.png')